## ▶ What you'll see when you run this
- The AI's **first** `best_price` patch is **wrong** — it crashes on the empty-coupons case — and your own test catches the bug, which you then fix.

**Time:** ~12 min · **Cost:** free (cheapest model: Gemini Flash / Claude Haiku / local Ollama) · **Keys:** ANTHROPIC_API_KEY

# Week 12 · Notebook 1 — The AI-as-Coder Loop
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Claude Code lives in your terminal — but the *idea* (reason → act → observe → review) works in a notebook too. Here we drive an LLM to write code against a **failing test**, exactly the way Lab A9 asks you to work, and we keep a human in the loop the whole time.

> Runs in Colab. **Degrades gracefully without an API key** — the fallback shows the same workflow with a hand-written patch.

## 0. Install + load key safely

In [ ]:
!pip -q install anthropic

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass  # locally: export ANTHROPIC_API_KEY=...
HAVE_KEY = bool(os.environ.get('ANTHROPIC_API_KEY'))
print('Anthropic key set:', HAVE_KEY)

## 1. The requirement, as a failing test
We want a function `best_price(price, coupons)` that applies the **single best** coupon (each is a percent-off int) and never returns a negative price. We capture that as a test *first*.

In [ ]:
TARGET = '''
def best_price(price, coupons):
    """Return price after applying the single best percent-off coupon.
    coupons: list of ints (percent off). Empty list => price unchanged.
    Never returns a negative number. Round to 2 decimals."""
    raise NotImplementedError
'''

TEST = '''
def run_tests(best_price):
    assert best_price(100, []) == 100
    assert best_price(100, [10, 25, 5]) == 75.0   # best = 25% off
    assert best_price(50, [200]) == 0.0           # clamp at 0
    assert best_price(19.99, [10]) == 17.99
    return 'all tests passed'
'''
print(TARGET); print(TEST)

## 2. Ask Claude to make the test pass
Note the prompt: **context** (the stub), **constraints** (signature, no negatives, rounding), **acceptance criteria** (these asserts). This is the Section-3 recipe from the lecture.

In [ ]:
def ask_claude_for_code(target, test):
    import anthropic
    client = anthropic.Anthropic()
    prompt = (
        'Implement the function below so it passes the tests. '
        'Keep the exact signature. Return ONLY a python code block.\n\n'
        'STUB:\n' + target + '\nTESTS:\n' + test
    )
    msg = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=600,
        messages=[{'role': 'user', 'content': prompt}])
    return msg.content[0].text

def extract_code(text):
    if '```' in text:
        text = text.split('```')[1]
        if text.startswith('python'):
            text = text[len('python'):]
    return text.strip()

# NOTE: this fallback is the AI's *first* attempt — and it is BUGGY on purpose.
# It forgets the empty-coupons case, so max([]) raises ValueError. Your job
# (Section 3 below) is to CATCH that with the test and fix it.
FALLBACK = '''
def best_price(price, coupons):
    best = max(coupons)              # BUG: crashes when coupons == []
    result = price * (1 - best / 100)
    return round(max(result, 0.0), 2)
'''

if HAVE_KEY:
    code = extract_code(ask_claude_for_code(TARGET, TEST))
    print('--- model wrote ---')
else:
    code = FALLBACK.strip()
    print('--- no key: using fallback patch (same workflow) ---')
print(code)

## 3. Review, then run the test yourself
**Never trust green you didn't run.** We exec the model's code in a scratch namespace and run the asserts. Read the code above *before* you run this.

> **Notice the AI's first attempt is wrong** — your job is to catch it with tests and fix it. The cell below is wrapped in `try/except`: when the first patch crashes on `best_price(100, [])`, that failure is the signal. Patch `code` to add the empty-coupons guard (`if not coupons: ...`), then re-run until you see `all tests passed`.

In [ ]:
ns = {}
exec(code, ns)             # define best_price
exec(TEST, ns)             # define run_tests
try:
    print(ns['run_tests'](ns['best_price']))
except Exception as e:
    print('Test FAILED — the AI got it wrong:', repr(e))
    print('Fix the bug in `code` above (handle empty coupons), then re-run.')

## 4. Add YOUR own edge case
The AI passed *our* tests. Did it handle a 0% coupon? Duplicate coupons? Add an assert the model wasn't given and re-run. This is the human-review step you'll repeat in Lab A9.

In [ ]:
def my_extra_tests(best_price):
    assert best_price(100, [0]) == 100      # 0% off
    assert best_price(100, [10, 10]) == 90  # duplicates
    # TODO: add one more of your own
    return 'my extra tests passed'

print(my_extra_tests(ns['best_price']))

## 5. Reflection (for A9)
In ~200 words: where did the AI help most, did it over-reach or hallucinate, and what did *you* have to verify? Then go do the real lab in `code/starter_repo/` with **Claude Code**.

In [ ]:
# notes:
